In [ ]:
import tkinter as tk

board = [[" " for _ in range(3)] for _ in range(3)]
buttons = [[None for _ in range(3)] for _ in range(3)]
game_over = False

def check_winner():
    for row in board:
        if row[0] == row[1] == row[2] != " ":
            return row[0]
    for col in range(3):
        if board[0][col] == board[1][col] == board[2][col] != " ":
            return board[0][col]
    if board[0][0] == board[1][1] == board[2][2] != " ":
        return board[0][0]
    if board[0][2] == board[1][1] == board[2][0] != " ":
        return board[0][2]
    return None

def is_full():
    for row in board:
        if " " in row:
            return False
    return True

def get_moves():
    moves = []
    for i in range(3):
        for j in range(3):
            if board[i][j] == " ":
                moves.append((i, j))
    return moves

def heuristic_evaluation():
    score = 0
    lines = []
    for i in range(3):
        lines.append([board[i][0], board[i][1], board[i][2]])
        lines.append([board[0][i], board[1][i], board[2][i]])
    lines.append([board[0][0], board[1][1], board[2][2]])
    lines.append([board[0][2], board[1][1], board[2][0]])
    
    for line in lines:
        if line.count("O") == 2 and line.count(" ") == 1:
            score += 10
        if line.count("X") == 2 and line.count(" ") == 1:
            score -= 10
    return score

def minimax(is_maximizing, alpha, beta, depth, use_pruning=True):
    winner = check_winner()
    if winner == "O": return 100 - depth
    if winner == "X": return -100 + depth
    if is_full(): return 0
    
    if depth >= 3:
        return heuristic_evaluation()

    moves = get_moves()
    if is_maximizing:
        best = -999
        for i, j in moves:
            board[i][j] = "O"
            score = minimax(False, alpha, beta, depth + 1, use_pruning)
            board[i][j] = " "
            best = max(best, score)
            if use_pruning:
                alpha = max(alpha, best)
                if beta <= alpha: break
        return best
    else:
        best = 999
        for i, j in moves:
            board[i][j] = "X"
            score = minimax(True, alpha, beta, depth + 1, use_pruning)
            board[i][j] = " "
            best = min(best, score)
            if use_pruning:
                beta = min(beta, best)
                if beta <= alpha: break
        return best

def ai_move():
    best_score = -999
    move = None
    moves = get_moves()
    for i, j in moves:
        board[i][j] = "O"
        score = minimax(False, -999, 999, 0, True)
        board[i][j] = " "
        if score > best_score:
            best_score = score
            move = (i, j)
    if move:
        i, j = move
        board[i][j] = "O"
        buttons[i][j].config(text="O", fg="#ff1493")

def click(i, j):
    global game_over
    if game_over or board[i][j] != " ": return
    board[i][j] = "X"
    buttons[i][j].config(text="X", fg="#d1006c")
    if check_winner() == "X":
        end_game("You Win!")
        return
    if is_full():
        end_game("Draw")
        return
    ai_move()
    if check_winner() == "O":
        end_game("AI Wins")
        return
    if is_full():
        end_game("Draw")
        return

def reset():
    global board, game_over
    game_over = False
    board = [[" " for _ in range(3)] for _ in range(3)]
    result_label.config(text="")
    for i in range(3):
        for j in range(3):
            buttons[i][j].config(text=" ")

def end_game(msg):
    global game_over
    game_over = True
    result_label.config(text=msg)
    window.after(1500, reset)

window = tk.Tk()
window.title("Tic Tac Toe AI")
window.configure(bg="#ffc0cb")
result_label = tk.Label(window, text="", font=("Arial", 16), bg="#ffc0cb", fg="#d1006c")
result_label.grid(row=0, column=0, columnspan=3)

for i in range(3):
    for j in range(3):
        buttons[i][j] = tk.Button(window, text=" ", font=("Arial", 20), width=5, height=2,
                                bg="#ffb6c1", command=lambda r=i, c=j: click(r, c))
        buttons[i][j].grid(row=i+1, column=j)

tk.Button(window, text="Reset", command=reset, bg="#ff69b4", fg="white").grid(row=4, column=0, columnspan=3)
window.mainloop()